[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ushaloy/PINN-LEWS-HYDROL86065/blob/main/notebooks/04_manuscript_figures_tables.ipynb)

# Manuscript figures and tables — HYDROL86065

**Run all cells.** This notebook redraws the data figures and rebuilds the data tables of the manuscript, with the **same figure and table numbers**, from the data and stored results in this repository. Nothing is trained. It takes a few minutes and needs no GPU and no Google Drive.

| Output | Folder | Format |
|---|---|---|
| Fig. 3, 4, 6–15 | `manuscript_figures/` (`Fig03.png` … `Fig15.png`) | PNG, 450 dpi, Times New Roman (Liberation Serif if Times New Roman is missing) |
| Table 2–7 | `manuscript_tables/` (`Table2.csv` … `Table7.csv`, `manuscript_tables.xlsx`) | CSV / Excel |

Not redrawn: Fig. 1 (map), Fig. 2 and Fig. 5 (schematics), Table 1 (site and model parameters) and Tables 8–9 (literature), because they are not computed from the data.

Where the notebook recomputes a number that is also stored in the repository, it checks the two against each other and prints `OK` or `CHECK`.

In [ ]:
import os, sys, io, json, re, subprocess, tempfile, warnings, contextlib
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

REPO_NAME = "PINN-LEWS-HYDROL86065"
REPO_URL  = "https://github.com/Ushaloy/PINN-LEWS-HYDROL86065"
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    REPO = os.path.join("/content", REPO_NAME)
    if not os.path.isdir(os.path.join(REPO, "data")):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
else:   # run from the repository's notebooks/ folder
    here = os.getcwd()
    REPO = os.path.dirname(here) if os.path.basename(here) == "notebooks" else here
assert os.path.isdir(os.path.join(REPO, "data")), "data/ not found: run this notebook inside the repository"
os.environ["PINN_REPO_DIR"] = REPO

RES = os.path.join(REPO, "results")
RC, RS = os.path.join(RES, "chrono"), os.path.join(RES, "season")
RB, RX = os.path.join(RES, "baselines"), os.path.join(RES, "external", "reference")
FIG_DIR = os.path.join(REPO, "manuscript_figures")
TAB_DIR = os.path.join(REPO, "manuscript_tables")
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TAB_DIR, exist_ok=True)
DPI = 450                      # resolution of every figure
RECOMPUTE_SOLVER = False       # True: re-run the Richards solver for Fig. 8 (about 1 min) instead of reading the stored series

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
try:
    from IPython.display import display
except ImportError:
    display = print

# --- font: Times New Roman, or its metric-compatible substitute Liberation Serif ---
def _has(name): return any(f.name == name for f in fm.fontManager.ttflist)
FONT = next((n for n in ("Times New Roman", "Liberation Serif", "Tinos", "Nimbus Roman") if _has(n)), None)
if FONT is None and IN_COLAB:
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-liberation"], capture_output=True)
    fm._load_fontmanager(try_read_cache=False)
    FONT = "Liberation Serif" if _has("Liberation Serif") else None
if FONT is None:
    FONT = "DejaVu Serif"; print("WARNING: no Times-like font found, using DejaVu Serif")
print("font:", FONT, "| dpi:", DPI)

plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "mathtext.fontset": "stix", "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8,
    "axes.labelsize": 9.5, "axes.titlesize": 10, "axes.titleweight": "normal", "xtick.major.width": 0.8, "ytick.major.width": 0.8,
    "legend.frameon": False, "legend.fontsize": 8, "figure.dpi": 110, "savefig.dpi": DPI,
    "pdf.fonttype": 42, "axes.axisbelow": True,
})
C = dict(piml="#1F5FA8", do="#D1495B", persist="#555555", arx="#8D99AE", rf="#BC6C25", gb="#DDA15E",
         lstm="#6A994E", lstm2="#A7C957", solver="#6D597A", solver0="#B5838D", grey="#6B7280",
         train="#E8EEF8", val="#FDF0D5", test="#F7D9DB", green="#6A994E", purple="#7B2CBF")
MS = 1e3      # m3 m-3  ->  x 10^-3 m3 m-3
UNIT = r"($\times10^{-3}$ m$^{3}$ m$^{-3}$)"

def save_fig(fig, num):
    path = os.path.join(FIG_DIR, f"Fig{num:02d}.png")
    fig.savefig(path, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.show(); print(f"saved {os.path.relpath(path, REPO)}")

def ygrid(ax): ax.grid(axis="y", color="#dddddd", lw=0.6)
def plabel(ax, s, x=-0.14, y=1.06): ax.text(x, y, s, transform=ax.transAxes, fontweight="bold", fontsize=11, va="top", ha="left")
def sdev(x): x = np.asarray(x, float); return float(np.std(x, ddof=1)) if len(x) > 1 else 0.0
def gap_break(t, y, gap_h=6.0):
    """Insert NaN where the time gap exceeds gap_h hours, so that outages are left blank."""
    t = pd.to_datetime(pd.Series(np.asarray(t))).reset_index(drop=True); y = pd.Series(np.asarray(y, float)).reset_index(drop=True)
    big = t.diff().dt.total_seconds() > gap_h * 3600
    idx = np.flatnonzero(big.values)
    tt = t.copy(); yy = y.copy()
    for i in idx[::-1]:
        tt = pd.concat([tt.iloc[:i], pd.Series([t.iloc[i - 1] + pd.Timedelta(seconds=1)]), tt.iloc[i:]], ignore_index=True)
        yy = pd.concat([yy.iloc[:i], pd.Series([np.nan]), yy.iloc[i:]], ignore_index=True)
    return tt.values, yy.values
def month_axis(ax, every=1, fmt="%b\n%Y"):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=every)); ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))

In [ ]:
# ---- data: the same cleaning code as notebook 01 (data preparation only, no training) ----
def load_namespace(split="chrono"):
    nb = json.load(open(os.path.join(REPO, "notebooks", "01_main_chronological.ipynb"), encoding="utf-8"))
    cells = ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
    os.environ["PINN_RESULTS_DIR"] = tempfile.mkdtemp()          # scratch folder; stored results are not touched
    g = {"__name__": "__main__", "display": lambda *a, **k: None}
    s0 = (cells[0].replace('SPLIT_MODE = "chrono"', f'SPLIT_MODE = "{split}"').replace("PREP_ONLY = False", "PREP_ONLY = True")
                  .replace("\nREUSE_STORED = True\n", "\nREUSE_STORED = False\n"))
    with contextlib.redirect_stdout(io.StringIO()):
        exec(compile(s0, "setup", "exec"), g)
        try: exec(compile(cells[1], "main", "exec"), g)
        except SystemExit: pass
    return g

G = load_namespace("chrono")
VG = G["VG"]
def run_prepare(csv, dev, warm):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        d = G["prepare_site"](csv, dev, warmup_theta=warm, label=f"Dev{dev}")
    return d, buf.getvalue()
d108, log108 = run_prepare(G["DATA_CSV"], 108, G["WARMUP_THETA_OK"])
d107, log107 = run_prepare(os.path.join(REPO, "data", "dev_107_updated.csv"), 107, None)
print(log108.strip()); print(log107.strip())
print("Dev108 chronological split:", {k: (str(G[k]["timestamp"].min())[:10], str(G[k]["timestamp"].max())[:10]) for k in ("df_tr", "df_val", "df_te")})

## Fig. 3 — Monitoring record, November 2025 – September 2026

In [ ]:
def daily_rain(d): return d.set_index("timestamp")["rain_inc"].resample("D").sum()
r108, r107 = daily_rain(d108), daily_rain(d107)
fig, ax = plt.subplots(3, 1, figsize=(7.4, 4.7), sharex=True, gridspec_kw=dict(height_ratios=[0.9, 1.3, 1.3], hspace=0.14))
off = pd.Timedelta(hours=5)
ax[0].bar(r108.index - off, r108.values, width=0.42, color=C["piml"], label="Dev108")
ax[0].bar(r107.index + off, r107.values, width=0.42, color="#E09F3E", label="Dev107")
ax[0].set_ylabel("Daily rain (mm)"); ax[0].legend(ncol=2, loc="upper right"); ygrid(ax[0])
spans = [(G["df_tr"], C["train"]), (G["df_val"], C["val"]), (G["df_te"], C["test"])]
for a, d, col, lab in [(ax[1], d108, C["piml"], "Dev108"), (ax[2], d107, "#E09F3E", "Dev107")]:
    for sp, c in spans: a.axvspan(sp["timestamp"].min(), sp["timestamp"].max(), color=c, lw=0)
    m = ~d["held"].values
    t, y = gap_break(d["timestamp"][m], d["theta"][m], gap_h=6)
    a.plot(t, y, color=col, lw=0.8); a.set_ylabel(f"{lab} " + r"$\theta$ (m$^{3}$ m$^{-3}$)"); ygrid(a)
    a.axvline(pd.Timestamp("2026-05-01"), color="k", ls="--", lw=0.9)
for key, lab in (("df_tr", "train"), ("df_val", "validation"), ("df_te", "test")):
    s = G[key]; ax[1].text(s["timestamp"].min() + (s["timestamp"].max() - s["timestamp"].min()) / 2 if key != "df_tr" else pd.Timestamp("2026-03-12"),
                           ax[1].get_ylim()[1] - 0.012, lab, ha="center", va="top", fontsize=8, color="#444444")
ax[2].text(pd.Timestamp("2026-05-03"), ax[2].get_ylim()[1] - 0.004, "season split: test from 1 May", va="top", fontsize=8)
month_axis(ax[2]); ax[2].set_xlim(pd.Timestamp("2025-10-24"), pd.Timestamp("2026-10-06"))
for a, s in zip(ax, "abc"): plabel(a, f"({s})", x=-0.16, y=1.12 if s == "a" else 1.05)
save_fig(fig, 3)

## Fig. 4 — Storm-scale response at 2-min resolution

In [ ]:
STORMS = [("18\u201320 Nov 2025", "2025-11-19 02:00", "2025-11-20 23:00"),
          ("13\u201315 May 2026", "2026-05-13 12:00", "2026-05-16 00:00"),
          ("12\u201314 Sep 2026", "2026-09-12 12:00", "2026-09-15 00:00")]
fig, axs = plt.subplots(2, 3, figsize=(7.4, 3.6), sharex="col", gridspec_kw=dict(height_ratios=[1, 2.3], hspace=0.1, wspace=0.34))
for j, (title, t0, t1) in enumerate(STORMS):
    t0, t1 = pd.Timestamp(t0), pd.Timestamp(t1)
    for d, col, o in ((d108, C["piml"], -1), (d107, "#E09F3E", 1)):
        w = d[(d["timestamp"] >= t0) & (d["timestamp"] <= t1)]
        half = w[w["timestamp"].dt.minute.isin([0, 30])]
        axs[0, j].bar(half["timestamp"] + o * pd.Timedelta(minutes=3.5), half["rain"].values, width=0.0045, color=col)
        mm = ~w["held"].values
        t, y = gap_break(w["timestamp"][mm], w["theta"][mm], gap_h=0.5)
        axs[1, j].plot(t, y, color=col, lw=1.0, label="Dev108" if o < 0 else "Dev107")
    axs[0, j].set_title(title, fontsize=9.5); axs[0, j].set_xlim(t0, t1); ygrid(axs[0, j]); ygrid(axs[1, j])
    axs[1, j].xaxis.set_major_locator(mdates.DayLocator()); axs[1, j].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    plabel(axs[0, j], f"({'abc'[j]})", x=-0.3 if j == 0 else -0.2, y=1.3)
axs[0, 0].set_ylabel("Rain (mm / 30 min)"); axs[1, 0].set_ylabel(r"$\theta$ at 0.30 m (m$^{3}$ m$^{-3}$)")
axs[1, 0].legend(loc="lower right", fontsize=8)
save_fig(fig, 4)

## Fig. 6 — Model selection

In [ ]:
# persistence RMSE on the validation set (reference line of panels b, c)
VAL_PERS = float(np.sqrt(np.mean((G["theta_val"] - G["X_val_raw"][:, 1]) ** 2)))
arch = pd.read_csv(os.path.join(RES, "architecture_search", "arch_stage2.csv"))
a = (arch.groupby(["depth", "width", "act", "taper"]).val_rmse.agg(["mean", lambda x: np.std(x, ddof=1)]).reset_index()
         .rename(columns={"<lambda_0>": "sd"}).sort_values("mean").reset_index(drop=True))
a["label"] = [f"{r.depth}\u00d7{r.width}\n+{r.width // 2}\n{r.act}" for r in a.itertuples()]
sens = pd.read_csv(os.path.join(RC, "sensitivity_lambda.csv"))
do_val = float(sens.loc[sens.param == "DATA_ONLY_all_lambda_0", "val_rmse"].iloc[0]) * MS

fig, ax = plt.subplots(1, 3, figsize=(8.6, 2.7), gridspec_kw=dict(wspace=0.62, width_ratios=[1.3, 1, 1]))
cols = [C["piml"] if (r.depth, r.width) == (5, 128) else (C["do"] if (r.depth, r.width) == (3, 128) else "#999999") for r in a.itertuples()]
ax[0].bar(range(len(a)), a["mean"] * MS, yerr=a["sd"] * MS, color=cols, width=0.82, capsize=3, error_kw=dict(lw=1))
ax[0].set_xticks(range(len(a))); ax[0].set_xticklabels(a["label"], fontsize=7)
ax[0].set_ylabel("Validation RMSE " + UNIT); ygrid(ax[0])
i5 = a.index[(a.depth == 5) & (a.width == 128)][0]; i3 = a.index[(a.depth == 3) & (a.width == 128)][0]
ax[0].text(i5, (a["mean"][i5] + a["sd"][i5]) * MS + 0.02, "selected", color=C["piml"], ha="center", fontsize=8)
ax[0].text(i3, (a["mean"][i3] + a["sd"][i3]) * MS + 0.02, "submitted", color=C["do"], ha="center", fontsize=8)
lam = sens[sens.param == "LAM"].sort_values("value"); lbc = sens[sens.param == "LAM_BC"].sort_values("value")
def lam_panel(a_, d, labels, xl, logy=False):
    x = np.arange(len(d)); y, e = d["val_rmse"].values * MS, d["val_rmse_std"].values * MS
    a_.errorbar(x, y, yerr=e, color=C["piml"], marker="o", ms=5, lw=1.4, capsize=3, label="PIML")
    k = int(np.flatnonzero(d["is_default"].values)[0]); a_.plot(x[k], y[k], "o", mfc="none", mec="k", ms=14, mew=1.2, label="final")
    a_.axhline(do_val, color=C["do"], ls="--", lw=1.1, label="data-only"); a_.axhline(VAL_PERS * MS, color=C["persist"], ls=":", lw=1.2, label="persistence")
    a_.set_xticks(x); a_.set_xticklabels(labels); a_.set_xlabel(xl); ygrid(a_)
    if logy:
        a_.set_yscale("log"); a_.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter()); a_.set_yticks([0.3, 0.5, 1, 2, 3]); a_.set_yticklabels(["0.3", "0.5", "1", "2", "3"]); a_.set_ylim(0.3, 3.6)
        a_.set_ylabel("Validation RMSE ($\\times10^{-3}$, log scale)")
    else:
        a_.set_ylim(0.2, 1.3); a_.set_ylabel("Validation RMSE " + UNIT)
lam_panel(ax[1], lam, ["0.001", "0.1", "1"], r"$\lambda_{phys}$")
lam_panel(ax[2], lbc, ["0", "0.001", "0.01", "0.1"], r"$\lambda_{BC}$", logy=True)
h, l = ax[1].get_legend_handles_labels(); order = [l.index(k) for k in ("final", "data-only", "persistence", "PIML")]
ax[1].legend([h[i] for i in order], [l[i] for i in order], loc="lower right", fontsize=7.5)
for a_, s, xx in zip(ax, "abc", (-0.26, -0.26, -0.34)): plabel(a_, f"({s})", x=xx, y=1.08)
save_fig(fig, 6)

## Results used by Fig. 7, 9, 13 and Table 3

In [ ]:
ml = pd.read_csv(os.path.join(RB, "ml_baselines.csv"))
lstm = pd.read_csv(os.path.join(RC, "lstm_results.csv"))
ms_ = {"chrono": pd.read_csv(os.path.join(RC, "multiseed_summary.csv")), "season": pd.read_csv(os.path.join(RS, "multiseed_summary.csv"))}
raw_chk = {"chrono": pd.read_csv(os.path.join(RC, "leakage_check_raw_target.csv")), "season": pd.read_csv(os.path.join(RS, "leakage_check_raw_target.csv"))}
phys = {(sp, m): pd.read_csv(os.path.join(RB, f"phys_{sp}_{m}.csv")).iloc[0] for sp in ("chrono", "season") for m in ("free", "onestep")}
PERS = {sp: float(ml[(ml.split == sp) & (ml.model == "persistence")].test_rmse.iloc[0]) for sp in ("chrono", "season")}

def method_rows(split):
    """One row per method: per-seed RMSE (x1e-3), mean, SD, R2 and RMSE against the raw readings."""
    rows = []
    def add(label, key, color, rmse, r2=np.nan, raw=np.nan):
        rmse = np.asarray(rmse, float) * MS
        rows.append(dict(label=label, key=key, color=color, rmse=rmse, mean=rmse.mean(), sd=sdev(rmse), r2=r2, raw=raw))
    g = lambda m: ml[(ml.split == split) & (ml.model == m)]
    p = g("persistence"); add("Persistence", "persistence", C["persist"], p.test_rmse, p.test_r2.mean(), p.test_rmse_raw.mean() * MS)
    x = g("ARX (ridge)"); add("ARX (ridge)", "arx", C["arx"], x.test_rmse, x.test_r2.mean(), x.test_rmse_raw.mean() * MS)
    x = g("Random forest"); add("Random forest", "rf", C["rf"], x.test_rmse, x.test_r2.mean(), x.test_rmse_raw.mean() * MS)
    x = g("Gradient boosting"); add("Gradient boosting", "gb", C["gb"], x.test_rmse, x.test_r2.mean(), x.test_rmse_raw.mean() * MS)
    if split == "chrono":
        for var, lab, col in (("current", "LSTM (current R)", C["lstm"]), ("history", "LSTM (R history)", C["lstm2"])):
            x = lstm[lstm.variant == var]; add(lab, "lstm_" + var, col, x.test_rmse, x.test_r2.mean())
    m = ms_[split]; rc = raw_chk[split].set_index("model")
    add("Data-only network", "ml", C["do"], m.ml_rmse, m.ml_r2.mean(), rc.loc["ML (main run)", "rmse_raw"] * MS)
    add("PIML (this study)", "piml", C["piml"], m.piml_rmse, m.piml_r2.mean(), rc.loc["PIML (main run)", "rmse_raw"] * MS)
    s1, s0 = phys[(split, "onestep")], phys[(split, "free")]
    add("Richards solver,\nIoT update every step", "solver", C["solver"], [s1.test_rmse], s1.test_r2, s1.test_rmse_raw * MS)
    add("Richards solver,\nno IoT update", "solver0", C["solver0"], [s0.test_rmse], s0.test_r2, s0.test_rmse_raw * MS)
    return rows
MR = {sp: method_rows(sp) for sp in ("chrono", "season")}
print({sp: [(r["key"], round(r["mean"], 3)) for r in MR[sp]] for sp in MR})

## Fig. 7 — Comparison with relevant methods

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.6, 3.4), gridspec_kw=dict(wspace=0.62))
for a_, sp, title in ((ax[0], "chrono", "Chronological test (12\u201329 Sep 2026)"), (ax[1], "season", "Out-of-season test (May\u2013Sep 2026)")):
    rows = MR[sp]; y = np.arange(len(rows))[::-1]
    for yi, r in zip(y, rows):
        a_.barh(yi, r["mean"], height=0.62, color=r["color"], xerr=r["sd"] if r["sd"] > 0 else None, error_kw=dict(lw=1.2, capsize=2.5, capthick=1.2))
        a_.text(max(r["mean"] + r["sd"], r["mean"]) * 1.08, yi, f"{r['mean']:.2f}", va="center", fontsize=8)
    a_.set_yticks(y); a_.set_yticklabels([r["label"] for r in rows], fontsize=8)
    a_.axvline(MR[sp][0]["mean"], color=C["persist"], ls=":", lw=1.2); a_.set_xscale("log")
    a_.set_xticks([0.3, 1, 3, 10, 30]); a_.set_xticklabels(["0.3", "1", "3", "10", "30"]); a_.set_xlim(0.2, 120)
    a_.set_xlabel("Test RMSE " + UNIT.replace(")", "), log scale")); a_.set_title(title, fontsize=9.5)
    a_.grid(axis="x", color="#dddddd", lw=0.6); a_.tick_params(axis="y", length=3)
plabel(ax[0], "(a)", x=-0.62, y=1.1); plabel(ax[1], "(b)", x=-0.62, y=1.1)
save_fig(fig, 7)

## Fig. 8 — Physics-only prediction (Richards solver, no IoT update)

In [ ]:
def solver_series():
    cached = os.path.join(RB, "phys_pred_chrono_free.csv")
    if os.path.exists(cached) and not RECOMPUTE_SOLVER:
        return pd.read_csv(cached, parse_dates=["timestamp"])
    out = tempfile.mkdtemp(); env = dict(os.environ, BASELINE_OUT=out, PINN_REPO_DIR=REPO)
    for cmd in (["prep_features.py", "chrono"], ["run_phys.py", "chrono", "free"]):
        subprocess.run([sys.executable] + cmd, cwd=os.path.join(REPO, "baselines"), env=env, check=True, capture_output=True)
    return pd.read_csv(os.path.join(out, "phys_pred_chrono_free.csv"), parse_dates=["timestamp"])
sv = solver_series()
dte = G["df_te"]; half = dte[dte["timestamp"].dt.minute.isin([0, 30]) & ~dte["held"].values]
fig, ax = plt.subplots(2, 1, figsize=(7.4, 3.6), sharex=True, gridspec_kw=dict(height_ratios=[1, 2.3], hspace=0.12))
ax[0].bar(half["timestamp"], half["rain"].values, width=0.0105, color=C["piml"]); ax[0].set_ylabel("Rain\n(mm / 30 min)"); ygrid(ax[0])
t, y = gap_break(sv["timestamp"], sv["obs"], gap_h=1); ax[1].plot(t, y, color="k", lw=1.1, label="IoT reading at 0.30 m (smoothed)")
t, y = gap_break(sv["timestamp"], sv["pred"], gap_h=1); ax[1].plot(t, y, color=C["solver0"], lw=1.4, ls="--", label="Richards solver, no IoT update (same parameters and domain)")
ax[1].set_ylabel(r"$\theta$ (m$^{3}$ m$^{-3}$)"); ax[1].legend(loc="lower left", fontsize=8); ygrid(ax[1])
ax[1].xaxis.set_major_locator(mdates.DayLocator(interval=2)); ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plabel(ax[0], "(a)", x=-0.12, y=1.15); plabel(ax[1], "(b)", x=-0.12, y=1.08)
rm = float(np.sqrt(np.nanmean((sv["obs"] - sv["pred"]) ** 2)))
print(f"solver test RMSE from the series: {rm*MS:.2f} x1e-3 | stored: {phys[('chrono','free')].test_rmse*MS:.2f}  ->", "OK" if abs(rm - phys[("chrono", "free")].test_rmse) < 1e-5 else "CHECK")
save_fig(fig, 8)

## Fig. 9 — Physics-informed versus data-only network

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8.8, 2.8), gridspec_kw=dict(wspace=0.6, width_ratios=[1.5, 1, 0.8]))
rows = {r["key"]: r for r in MR["chrono"]}
bars = [("piml", "PIML", C["piml"]), ("ml", "Data-only", C["do"]), ("lstm_current", "LSTM\n(current R)", C["lstm"]), ("lstm_history", "LSTM\n(R history)", C["lstm2"])]
for i, (k, lab, col) in enumerate(bars):
    r = rows[k]; ax[0].bar(i, r["mean"], width=0.68, color=col, alpha=0.9)
    ax[0].errorbar(i, r["mean"], yerr=r["sd"], color="k", capsize=4, lw=1.2)
    ax[0].scatter(np.full(len(r["rmse"]), i) + np.array([-0.12, 0.0, 0.12]) * (len(r["rmse"]) == 3), r["rmse"], color="k", s=14, zorder=5)
ax[0].axhline(PERS["chrono"] * MS, color=C["persist"], ls=":", lw=1.4); ax[0].text(3.45, PERS["chrono"] * MS + 0.008, "persistence", ha="right", fontsize=7.5, color=C["persist"])
ax[0].set_xticks(range(4)); ax[0].set_xticklabels([b[1] for b in bars], fontsize=7.5); ax[0].set_ylabel("Test RMSE " + UNIT); ygrid(ax[0])
rc = raw_chk["chrono"].set_index("model"); names = [("Persistence", "Persistence (lag-1 input)"), ("PIML", "PIML (main run)"), ("Data-only", "ML (main run)")]
w = 0.36
for i, (lab, key) in enumerate(names):
    ax[1].bar(i - w / 2, rc.loc[key, "rmse_smoothed"] * MS, w, color="#9FB3D6", label="smoothed target" if i == 0 else None)
    ax[1].bar(i + w / 2, rc.loc[key, "rmse_raw"] * MS, w, color="#1F3A66", label="raw reading" if i == 0 else None)
ax[1].set_xticks(range(3)); ax[1].set_xticklabels([n[0] for n in names], fontsize=8); ax[1].set_ylabel("Test RMSE " + UNIT); ax[1].set_ylim(0, 0.8)
ax[1].legend(loc="upper left", fontsize=7.5); ygrid(ax[1])
res_p = float(sens.loc[sens.param == "LAM", :].query("is_default").val_phys.iloc[0]); res_d = float(sens.loc[sens.param == "DATA_ONLY_all_lambda_0", "val_phys"].iloc[0])
for i, (v, col, lab) in enumerate(((res_p, C["piml"], "PIML"), (res_d, C["do"], "Data-only"))):
    ax[2].bar(i, v, width=0.62, color=col); ax[2].text(i, v * 1.6, f"{v:.1e}".replace("e-0", "e-0"), ha="center", fontsize=8)
ax[2].set_yscale("log"); ax[2].set_ylim(1e-6, 1e2); ax[2].set_xticks([0, 1]); ax[2].set_xticklabels(["PIML", "Data-only"], fontsize=8)
ax[2].set_ylabel("Normalised Richards residual (val.)"); ygrid(ax[2])
for a_, s in zip(ax, "abc"): plabel(a_, f"({s})", x=-0.26, y=1.1)
print(f"Richards residual (validation inputs, one-third-budget runs): PIML {res_p:.2e} | data-only {res_d:.3f}")
save_fig(fig, 9)

## Fig. 10 — Lower-boundary and initial-condition scenarios

In [ ]:
bs = pd.read_csv(os.path.join(RC, "sensitivity_lower_bc_ic_summary.csv"), header=[0, 1]).iloc[1:].reset_index(drop=True)
bs.columns = ["lbc", "ic", "lam"] + ["_".join(c) for c in bs.columns[3:]]
for c in bs.columns[3:]: bs[c] = pd.to_numeric(bs[c], errors="coerce")
bs["lam"] = pd.to_numeric(bs["lam"])
bps = pd.read_csv(os.path.join(RC, "sensitivity_lower_bc_ic_per_seed.csv"))
met = bps.groupby(["lbc", "ic", "lam_lbc"]).bc_met.sum().rename("n_met").reset_index()
def scen_label(r):
    return {"free_drainage": "FD", "no_flux": "NF", "water_table": "WT"}[r.lbc] + ("\nsensor\nIC" if (r.lbc == "free_drainage" and r.ic == "sensor") else ("\nuniform\nIC" if r.lbc == "free_drainage" else f"\n\u03bb = {r.lam:g}"))
bs["label"] = [scen_label(r) for r in bs.itertuples()]
bs["all_met"] = [bool(met[(met.lbc == r.lbc) & (met.ic == r.ic) & (np.isclose(met.lam_lbc, r.lam))].n_met.iloc[0] == 3) for r in bs.itertuples()]
fig, ax = plt.subplots(1, 2, figsize=(8.2, 2.9), gridspec_kw=dict(wspace=0.3))
x = np.arange(len(bs))
for i, r in bs.iterrows():
    ax[0].bar(i, r["val_rmse_mean"] * MS, width=0.8, facecolor=C["piml"] if r["all_met"] else "white", edgecolor=C["piml"], hatch=None if r["all_met"] else "///", lw=1.2)
    ax[0].errorbar(i, r["val_rmse_mean"] * MS, yerr=r["val_rmse_std"] * MS, color="k", capsize=4, lw=1.2)
ax[0].set_yscale("log"); ax[0].set_yticks([1, 2, 5, 10, 20, 50]); ax[0].set_yticklabels(["1", "2", "5", "10", "20", "50"]); ax[0].set_ylim(0.7, 90)
ax[0].set_xticks(x); ax[0].set_xticklabels(bs["label"], fontsize=7.5); ax[0].set_ylabel("Validation RMSE " + UNIT)
ax[0].text(0.6, 40, "hatched: condition\nnot met at 0.5 m", fontsize=7.5, va="center"); ygrid(ax[0])
for i, r in bs.iterrows():
    for off, col, mk in ((-0.17, "fos_min_sensor", "o"), (0.17, "fos_min_0p5m", "s")):
        c = (C["piml"] if mk == "o" else "#E09F3E") if r["all_met"] else ("#999999" if mk == "o" else "#CCCCCC")
        ax[1].errorbar(i + off, r[col + "_mean"], yerr=r[col + "_std"], fmt=mk, color=c, capsize=3, ms=5.5, lw=1.4)
ax[1].axhspan(1.66, 1.69, color="#E8EEF8", lw=0); ax[1].set_xticks(x); ax[1].set_xticklabels(bs["label"], fontsize=7.5)
ax[1].set_ylabel("Minimum FoS (test window)"); ygrid(ax[1])
ax[1].legend(handles=[Line2D([], [], marker="o", color=C["piml"], ls="", label=r"$\theta$ at sensor (0.30 m)"), Line2D([], [], marker="s", color="#E09F3E", ls="", label=r"$\theta$ at 0.50 m")], loc="upper left", fontsize=7.5)
for a_, s in zip(ax, "ab"): plabel(a_, f"({s})", x=-0.2, y=1.08)
save_fig(fig, 10)

## Fig. 11 — Transfer from Dev108 to Dev107

In [ ]:
t107 = pd.read_csv(os.path.join(RC, "transfer_107_summary.csv")); tml = pd.read_csv(os.path.join(RB, "transfer_ml.csv"))
PERS107, PERS107_RAW = float(t107[t107.strategy == "persistence"].test_rmse.iloc[0]), float(t107[t107.strategy == "persistence"].test_rmse_raw.iloc[0])
NDAYS = [3, 7, 14, 30]
STRAT = [("recal", "PIML + recalibration (2)", C["grey"], "D", "-"), ("head_pinn", "PIML, last layer (65)", C["piml"], "^", "-"),
         ("full_pinn", "PIML, all layers", C["piml"], "o", "-"), ("full_data_only", "data-only net, all layers", C["do"], "o", "-"),
         ("scratch_pinn", "PIML from scratch", C["piml"], "s", ":"), ("scratch_data_only", "data-only net from scratch", C["do"], "s", ":")]
def curve(df, strat):
    d = df[df.strategy == strat].set_index("n_days").reindex(NDAYS); return d["test_rmse"].values * MS, d["test_rmse_sd"].fillna(0).values * MS, d
fig, ax = plt.subplots(1, 2, figsize=(7.6, 3.3), gridspec_kw=dict(wspace=0.32))
handles = []
for strat, lab, col, mk, ls in STRAT:
    y, e, d = curve(t107, strat)
    h = ax[0].errorbar(NDAYS, y, yerr=e, color=col, marker=mk, ls=ls, ms=5, lw=1.4, capsize=3, label=lab); handles.append(h)
    sk = 1 - (d["test_rmse_raw"].values / PERS107_RAW) ** 2
    ax[1].plot(NDAYS, sk, color=col, marker=mk, ls=ls, ms=5, lw=1.4)
for key, lab, col, mk in (("ARX from scratch", "ARX from scratch", C["arx"], "v"), ("RF from scratch", "random forest from scratch", C["rf"], "P")):
    d = tml[tml.model == key].groupby("n_days")
    y = d.test_rmse.mean().reindex(NDAYS).values * MS; e = d.test_rmse.std(ddof=1).reindex(NDAYS).fillna(0).values * MS; sk = d.skill_raw.mean().reindex(NDAYS).values
    handles.append(ax[0].errorbar(NDAYS, y, yerr=e, color=col, marker=mk, ls="-.", ms=5.5, lw=1.4, capsize=3, label=lab)); ax[1].plot(NDAYS, sk, color=col, marker=mk, ls="-.", ms=5.5, lw=1.4)
ax[0].axhline(PERS107 * MS, color="#444444", ls="--", lw=1.1); ax[0].text(30, PERS107 * MS * 0.93, "persistence", ha="right", va="top", fontsize=7.5, color="#555555")
ax[0].set_yscale("log"); ax[0].set_yticks([1, 2, 5, 10]); ax[0].set_yticklabels(["1", "2", "5", "10"]); ax[0].set_ylim(0.78, 12)
ax[0].set_ylabel("Dev107 test RMSE " + UNIT)
ax[1].axhline(0, color="#444444", ls="--", lw=1.1); ax[1].set_yscale("symlog", linthresh=0.5); ax[1].set_ylim(-22, 1)
ax[1].set_yticks([1, 0.5, 0, -1, -5, -10, -20]); ax[1].set_yticklabels(["1", "0.5", "0", "\u22121", "\u22125", "\u221210", "\u221220"]); ax[1].set_ylabel("Skill vs persistence, raw readings")
for a_ in ax: a_.set_xticks(NDAYS); a_.set_xlabel("Days of Dev107 data used (N)"); ygrid(a_)
plabel(ax[0], "(a)", x=-0.17, y=1.08); plabel(ax[1], "(b)", x=-0.17, y=1.08)
fig.legend(handles=[h for h in handles], loc="lower center", ncol=4, fontsize=7.5, bbox_to_anchor=(0.5, -0.2))
save_fig(fig, 11)

## Fig. 12 — Transfer from Dev108 to sites in other countries

In [ ]:
ext = {"utuado": pd.read_csv(os.path.join(RX, "transfer_utuado_measured_soil.csv")), "california": pd.read_csv(os.path.join(RX, "transfer_california.csv"))}
ext_extra = {"range": pd.read_csv(os.path.join(RX, "transfer_utuado_range_only.csv")), "dev108": pd.read_csv(os.path.join(RX, "transfer_utuado_dev108_soil.csv"))}
fig = plt.figure(figsize=(7.8, 3.9)); gs = fig.add_gridspec(1, 3, wspace=0.36, width_ratios=[1, 1, 1.05]); ax = [fig.add_subplot(gs[i]) for i in range(3)]
for a_, (site, title) in zip(ax[:2], (("utuado", "Utuado (soil measured)"), ("california", "California (no soil data)"))):
    df = ext[site]; pers = float(df[df.strategy == "persistence"].test_rmse.iloc[0]) * MS
    for strat, lab, col, mk, ls in STRAT:
        y, e, _ = curve(df, strat); a_.errorbar(NDAYS, y, yerr=e, color=col, marker=mk, ls=ls, ms=5, lw=1.4, capsize=3)
    a_.plot(0, float(df[df.strategy == "zero_shot"].test_rmse.iloc[0]) * MS, marker="*", ms=13, color=C["piml"], ls="")
    a_.plot(0.4, float(df[df.strategy == "zero_shot_data_only"].test_rmse.iloc[0]) * MS, marker="*", ms=13, mfc="white", mec=C["do"], mew=1.3, ls="")
    a_.axhline(pers, color="#444444", ls="--", lw=1.1); a_.text(30, pers * 0.9, "persistence", ha="right", va="top", fontsize=7.5, color="#555555")
    a_.set_yscale("log"); a_.set_xticks([0] + NDAYS); a_.set_xlabel("Days of local data used (N)"); a_.set_title(title, fontsize=9.5); ygrid(a_)
ax[0].set_yticks([3, 4, 6, 10, 15]); ax[0].set_yticklabels(["3", "4", "6", "10", "15"]); ax[0].set_ylim(3, 17); ax[0].set_ylabel("Test RMSE " + UNIT)
ax[1].set_yticks([0.5, 1, 2, 5, 10, 20]); ax[1].set_yticklabels(["0.5", "1", "2", "5", "10", "20"]); ax[1].set_ylim(0.45, 26)
zs = lambda d: float(d[d.strategy == "zero_shot"].test_rmse.iloc[0]) / float(d[d.strategy == "persistence"].test_rmse.iloc[0])
bars = [("Dev107\n(same slope)", float(t107[t107.strategy == "zero_shot"].test_rmse.iloc[0]) / PERS107, C["green"]),
        ("Utuado,\nmeasured soil", zs(ext["utuado"]), C["piml"]), ("Utuado,\n$\\theta_s$, $\\theta_r$ only", zs(ext_extra["range"]), "#7BA7D7"),
        ("Utuado,\nDev108 soil", zs(ext_extra["dev108"]), "#B8CDE8"), ("California,\n$\\theta_s$, $\\theta_r$\nfrom record", zs(ext["california"]), "#999999")]
yy = np.arange(len(bars))[::-1]
for yi, (lab, v, col) in zip(yy, bars):
    ax[2].barh(yi, v, height=0.62, color=col); ax[2].text(v * 1.08, yi, f"{v:.1f}\u00d7", va="center", fontsize=8)
ax[2].set_yticks(yy); ax[2].set_yticklabels([b[0] for b in bars], fontsize=7.5); ax[2].set_xscale("log"); ax[2].axvline(1, color="#444444", ls="--", lw=1.1)
ax[2].set_xticks([1, 2, 5, 10, 20, 50]); ax[2].set_xticklabels(["1", "2", "5", "10", "20", "50"]); ax[2].set_xlim(0.8, 80)
ax[2].set_xlabel("Day-one RMSE / persistence RMSE"); ax[2].set_title("Day one, physics-informed", fontsize=9.5); ax[2].grid(axis="x", color="#dddddd", lw=0.6)
leg = [Line2D([], [], marker="*", ms=11, color=C["piml"], ls="", label="zero-shot, physics (day one)"), Line2D([], [], marker="*", ms=11, mfc="white", mec=C["do"], ls="", label="zero-shot, data-only (day one)")]
leg += [Line2D([], [], marker=mk, color=col, ls=ls, ms=5, lw=1.4, label=lab) for _, lab, col, mk, ls in STRAT]
for a_, s in zip(ax, "abc"): plabel(a_, f"({s})", x=-0.2, y=1.1)
fig.legend(handles=leg, loc="lower center", ncol=4, fontsize=7.5, bbox_to_anchor=(0.5, -0.16))
save_fig(fig, 12)

## Fig. 13 — Test RMSE of the physics-informed and data-only networks and persistence

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.4, 2.9), gridspec_kw=dict(wspace=0.34))
for a_, sp, title in ((ax[0], "chrono", "Chronological test (12\u201329 Sep)"), (ax[1], "season", "Out-of-season test (May\u2013Sep)")):
    rows = {r["key"]: r for r in MR[sp]}
    for i, (k, lab, col) in enumerate((("piml", "PIML", C["piml"]), ("ml", "Data-only", C["do"]), ("persistence", "Persistence", "#888888"))):
        r = rows[k]; a_.bar(i, r["mean"], width=0.6, color=col, alpha=0.95)
        if k != "persistence":
            a_.errorbar(i, r["mean"], yerr=r["sd"], color="k", capsize=4, lw=1.2)
            a_.scatter(i + np.array([-0.13, 0.0, 0.13]), r["rmse"], color="k", s=14, zorder=5)
    a_.set_xticks(range(3)); a_.set_xticklabels(["PIML", "Data-only", "Persistence"]); a_.set_title(title, fontsize=9.5)
    a_.set_ylabel("Test RMSE " + UNIT); ygrid(a_)
plabel(ax[0], "(a)", x=-0.2, y=1.1); plabel(ax[1], "(b)", x=-0.2, y=1.1)
save_fig(fig, 13)

## Fig. 14 — Slope stability at Dev108

In [ ]:
# FoS from the measured (smoothed) theta: Eq. (3) for the pressure head, u = Se * gamma_w * psi, Eq. (13) for the FoS
SLOPE, C_EFF, PHI, GAMMA, H, GAMMA_W, PSI_MIN = 33.0, 5.0, 28.0, 14.715, 1.0, 9.81, -165.0
def psi_se(theta):
    se = np.clip((np.asarray(theta, float) - VG["theta_r"]) / (VG["theta_s"] - VG["theta_r"]), 1e-6, 1 - 1e-9)
    m = 1 - 1 / VG["n"]; psi = -(1 / VG["alpha"]) * (se ** (-1 / m) - 1) ** (1 / VG["n"])
    return np.maximum(psi, PSI_MIN), se
def fos_from_theta(theta, beta=SLOPE, c=C_EFF, phi=PHI):
    psi, se = psi_se(theta); u = se * GAMMA_W * psi; b = np.radians(beta)
    return (c + (GAMMA * H * np.cos(b) ** 2 - u) * np.tan(np.radians(phi))) / (GAMMA * H * np.sin(b) * np.cos(b))
ft = pd.read_csv(os.path.join(RC, "fos_timeseries_main.csv"), parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
chk = np.nanmax(np.abs(fos_from_theta(ft["theta_piml"]) - ft["FoS_piml"]))
print(f"FoS recomputed from the stored PIML theta vs stored FoS: max difference {chk:.1e} ->", "OK" if chk < 1e-3 else "CHECK")
fos_obs = fos_from_theta(ft["theta_obs"]); psi_obs, _ = psi_se(ft["theta_obs"])
print(f"FoS from the measured theta: min {np.nanmin(fos_obs):.3f} | mean {np.nanmean(fos_obs):.3f}   (manuscript: 1.584 from the measurements, 1.589/1.590 from the PIML)")
fig = plt.figure(figsize=(7.8, 4.3)); gs = fig.add_gridspec(3, 2, width_ratios=[2.6, 1], hspace=0.2, wspace=0.28)
axa = [fig.add_subplot(gs[i, 0]) for i in range(3)]; axb = fig.add_subplot(gs[:, 1])
ser = [(ft["theta_obs"], C["piml"], r"$\theta$ (m$^{3}$ m$^{-3}$)"), (np.abs(psi_obs), C["purple"], r"$|\psi|$ (m)"), (fos_obs, C["green"], "FoS (\u2212)")]
for a_, (y, col, lab) in zip(axa, ser):
    t, yy = gap_break(ft["timestamp"], y, gap_h=6); a_.plot(t, yy, color=col, lw=0.8); a_.set_ylabel(lab); ygrid(a_)
axa[1].set_yscale("log"); axa[1].set_yticks([0.03, 0.1, 0.3, 1]); axa[1].set_yticklabels(["0.03", "0.1", "0.3", "1"])
axa[2].axhline(1.3, color=C["do"], ls="--", lw=1.1); axa[2].text(pd.Timestamp("2025-11-05"), 1.32, "FoS = 1.3", color=C["do"], fontsize=8, va="bottom"); axa[2].set_ylim(1.25, 2.05)
for a_ in axa[:2]: a_.tick_params(labelbottom=False)
axa[2].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 3, 5, 7, 9, 11])); axa[2].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
angles = np.arange(23, 42, 2); mn = [np.nanmin(fos_from_theta(ft["theta_obs"], beta=b)) for b in angles]; me = [np.nanmean(fos_from_theta(ft["theta_obs"], beta=b)) for b in angles]
axb.plot(angles, mn, color=C["green"], marker="o", ms=5, lw=1.5, label="minimum"); axb.plot(angles, me, color="#999999", marker="s", ms=5, lw=1.3, ls="--", label="mean")
axb.axvline(33, color="k", ls=":", lw=0.9); axb.text(33.3, max(me) * 0.99, "site\n33\u00b0", fontsize=8, va="top"); axb.axhline(1.3, color=C["do"], ls="--", lw=1.1)
axb.set_xlabel("Slope angle $\\beta$ (\u00b0)"); axb.set_ylabel("FoS (\u2212)"); axb.legend(loc="lower left"); ygrid(axb)
plabel(axa[0], "(a)", x=-0.12, y=1.25); plabel(axb, "(b)", x=-0.25, y=1.06)
print("min FoS vs slope angle (measured theta):", dict(zip(angles.tolist(), np.round(mn, 3).tolist())))
sl = pd.read_csv(os.path.join(RC, "fos_sensitivity_slope.csv")).set_index("slope_deg")
print("stored slope-sensitivity file (PIML theta) agrees within", f"{np.max(np.abs(sl.loc[angles, 'FoS_min'].values - np.array(mn))):.3f}", "(measured vs PIML series)")
save_fig(fig, 14)

## Fig. 15 — Alert stream of the final physics-informed network

In [ ]:
# Section 3.8: T_L = (FoS - FoS_crit) / |dFoS/dt| (Eq. 14), falling FoS only, capped at 48 h; same rules as notebook 01
FOSCRIT, TL_CAP_H, YELLOW_H, RED_H, SPINUP_H, OUTAGE_S, CADENCE_MIN, ESC_WIN, ESC_K, RELEASE_MIN, SE_MIN = 1.30, 48.0, 6.0, 2.0, 24.0, 3600.0, 2.0, 3, 2, 30.0, 0.50
def load_stream(f):
    f = f.sort_values("timestamp").drop_duplicates(subset="timestamp", keep="first").reset_index(drop=True)
    gap_s = f["timestamp"].diff().dt.total_seconds(); rate = f["FoS_piml"].diff() / (gap_s / 3600.0); decay = rate.clip(upper=0.0).abs()
    with np.errstate(divide="ignore", invalid="ignore"): TL = ((f["FoS_piml"] - FOSCRIT) / decay).clip(upper=TL_CAP_H)
    TL[(decay < 1e-12) | decay.isna()] = TL_CAP_H; f["TL"] = TL
    se = (f["theta_piml"] - VG["theta_r"]) / (VG["theta_s"] - VG["theta_r"])
    f["analysed"] = ((f["timestamp"] >= f["timestamp"].iloc[0] + pd.Timedelta(hours=SPINUP_H)) & ~(gap_s > OUTAGE_S) & (se >= SE_MIN)).values
    return f
def roll_ge(flags, w, k): return np.convolve(flags.astype(int), np.ones(w, dtype=int), mode="full")[:len(flags)] >= k
def debounced(red, nong):
    trig = roll_ge(nong, ESC_WIN, ESC_K); rel = int(round(RELEASE_MIN / CADENCE_MIN)); esc = np.zeros(len(red), bool); on, run = False, 0
    for i in range(len(red)):
        if not on and trig[i]: on, run = True, 0
        if on:
            run = run + 1 if not nong[i] else 0
            if run >= rel: on = False
        esc[i] = on
    return esc, red & esc
def episodes(flags):
    out, s = [], None
    for i, v in enumerate(flags):
        if v and s is None: s = i
        elif not v and s is not None: out.append((s, i - 1)); s = None
    if s is not None: out.append((s, len(flags) - 1))
    return out
fs = load_stream(ft); fa = fs[fs["analysed"]].reset_index(drop=True)
red = (fa["TL"] <= RED_H).values; nong = (fa["TL"] <= YELLOW_H).values; yel = nong & ~red
esc, dred = debounced(red, nong); e_red, e_esc, e_dred = episodes(red), episodes(esc), episodes(dred)
days = fa["timestamp"].diff().dt.total_seconds().clip(upper=OUTAGE_S).sum() / 86400.0
stored = open(os.path.join(RC, "alert_stream_summary.txt"), encoding="utf-8").read()
n_stored = int(re.search(r"analysed steps n\s*:\s*([\d,]+)", stored).group(1).replace(",", ""))
print(f"analysed samples {len(fa):,} (stored {n_stored:,}) | days {days:.1f} | raw RED episodes {len(e_red)} | escalations {len(e_esc)} | debounced RED {len(e_dred)} ->",
      "OK" if len(fa) == n_stored else "CHECK")
fig, ax = plt.subplots(2, 1, figsize=(8.0, 4.8), sharex=True, gridspec_kw=dict(height_ratios=[2, 1], hspace=0.08))
t, y = gap_break(fa["timestamp"], fa["FoS_piml"], gap_h=6); ax[0].plot(t, y, color="#555555", lw=0.7, label="FoS (PIML)")
ax[0].scatter(fa["timestamp"][yel], fa["FoS_piml"][yel], s=9, color="#F9A825", zorder=4, label="YELLOW (2 h < $T_L$ \u2264 6 h)")
ax[0].scatter(fa["timestamp"][red], fa["FoS_piml"][red], s=12, color="#C62828", zorder=5, label="RED ($T_L$ \u2264 2 h)")
for a0, b0 in e_esc: ax[0].axvspan(fa["timestamp"].iloc[a0], fa["timestamp"].iloc[b0], color="#F9A825", alpha=0.12, lw=0)
ax[0].axhline(FOSCRIT, color="#C62828", ls="--", lw=1.1, label="FoS = 1.3"); ax[0].set_ylabel("FoS (\u2212)"); ax[0].set_ylim(1.27, 1.93)
ax[0].set_title(f"Alert stream, analysed record (n = {len(fa):,}); shaded = confirmed escalations", loc="left", fontsize=9.5, fontweight="bold")
ax[0].legend(ncol=4, loc="upper right", fontsize=7.5); ax[0].grid(color="#dddddd", lw=0.5, ls="--")
t, y = gap_break(fa["timestamp"], fa["TL"], gap_h=6); ax[1].plot(t, y, color="#143F7D", lw=0.7)
ax[1].axhspan(RED_H, YELLOW_H, color="#F9A825", alpha=0.15, lw=0); ax[1].axhspan(0, RED_H, color="#C62828", alpha=0.12, lw=0)
ax[1].set_ylim(0, TL_CAP_H); ax[1].set_ylabel("$T_L$ (h)"); ax[1].grid(color="#dddddd", lw=0.5, ls="--")
ax[1].xaxis.set_major_locator(mdates.MonthLocator()); ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
save_fig(fig, 15)

## Tables 2–7\n\nEach table is shown, saved as `manuscript_tables/TableN.csv` and collected in `manuscript_tables.xlsx`.

In [ ]:
TABLES = {}
def keep(num, df, caption):
    TABLES[num] = df; df.to_csv(os.path.join(TAB_DIR, f"Table{num}.csv"), index=False, encoding="utf-8-sig")
    print(f"\nTable {num}. {caption}"); display(df)
f3 = lambda x, n=3: f"{x:.{n}f}"
pm = lambda m, s, n=3: f"{m:.{n}f} \u00b1 {s:.{n}f}" if s > 0 else f"{m:.{n}f}"

# ---- Table 2: records before and after processing ----
def parse_log(log, d):
    l1, l2 = log.strip().split("\n")[:2]
    num = lambda s: int(s.replace(",", ""))
    raw_rows = num(re.search(r"([\d,]+) raw rows", l1).group(1)); dup = num(re.search(r"([\d,]+) duplicates", l1).group(1))
    warm = num(re.search(r"warm-up (\d+) rows", l1).group(1)); meas = num(re.search(r"\| ([\d,]+) measured 2-min points", l1).group(1))
    fill = num(re.search(r"\(([\d,]+) filled-in", l1).group(1)); held = num(re.search(r"([\d,]+) short-outage", l1).group(1))
    rain = re.search(r"rain (\d+) mm", l2).group(1); big = re.search(r"max 30-min depth ([\d.]+) mm", l2).group(1)
    rng = re.search(r"\u03b8 ([\d.]+)\u2013([\d.]+)", l2); seg = re.search(r"(\d+) segments", l2).group(1); gap = re.search(r"longest gap ([\d.]+) d", l2).group(1)
    days_ = d.loc[~d["held"], "timestamp"].dt.normalize().nunique()
    return [f"{raw_rows:,}", f"{dup:,}", f"{warm:,}", f"{fill:,}", f"{meas:,}", f"{held:,}", f"{days_}", seg, gap, rain, big, f"{rng.group(1)}\u2013{rng.group(2)}"]
T2_rows = ["Raw records", "Duplicate time stamps removed", "Warm-up records removed", "Logger-infill values removed (2-min grid)", "Measured 2-min points",
           "Held points (inputs only)", "Days with data", "Continuous segments", "Longest outage (d)", "Total rain (mm)", "Largest 30-min rain (mm)", "\u03b8 range (m3 m-3)"]
keep(2, pd.DataFrame({"Quantity": T2_rows, "Dev108": parse_log(log108, d108), "Dev107": parse_log(log107, d107)}), "Records before and after processing.")

In [ ]:
# ---- Table 3: comparison with relevant methods at Dev108 ----
rn = {sp: MR[sp] for sp in MR}
def skill_per_seed(r, sp): return float(1 - (r["mean"] / (PERS[sp] * MS)) ** 2)      # skill = 1 - MSE/MSE_persistence, from the mean RMSE over seeds
rows3 = []
season = {r["key"]: r for r in rn["season"]}
for r in rn["chrono"]:
    s = season.get(r["key"]); key = r["key"]
    resid = {"piml": f"{res_p:.1e}", "ml": f"{res_d:.2f}", "solver": "0*", "solver0": "0*"}.get(key, "\u2013")
    rows3.append({"Model": r["label"].replace("\n", " "), "RMSE, chronological": pm(r["mean"], r["sd"]), "R2": f3(r["r2"], 4),
                  "Skill": f"{skill_per_seed(r, 'chrono'):.2f}", "RMSE vs raw": f3(r["raw"]) if np.isfinite(r["raw"]) else "\u2013",
                  "RMSE, out-of-season": pm(s["mean"], s["sd"], 2 if s["mean"] < 20 else 1) if s else "\u2013",
                  "Skill, out-of-season": f"{skill_per_seed(s, 'season'):.2f}" if s else "\u2013", "Richards residual": resid})
keep(3, pd.DataFrame(rows3), "Comparison with relevant methods at Dev108: test RMSE (x1e-3 m3 m-3), R2 and skill relative to persistence (skill from the mean RMSE over seeds).")

In [ ]:
# ---- Table 4: lower-boundary and initial-condition scenarios ----
ident = pd.read_csv(os.path.join(RC, "sensitivity_lower_bc_identifiability.csv"))
def lab4(r):
    nm = {"free_drainage": "Free drainage", "no_flux": "No-flux", "water_table": "Water table"}[r.lbc]
    return f"{nm}, {r.ic}, {r.lam:g}" + (" (final)" if (r.lbc == "free_drainage" and r.ic == "sensor") else "")
rows4 = []
for r in bs.itertuples():
    n = int(met[(met.lbc == r.lbc) & (met.ic == r.ic) & (np.isclose(met.lam_lbc, r.lam))].n_met.iloc[0])
    idr = ident[(ident.lbc == r.lbc) & (ident.ic == r.ic) & (np.isclose(ident.lam_lbc, r.lam))]
    dts = f"{idr.d_theta_sensor.iloc[0]:.4f}" if len(idr) else "\u2013"; dtb = f"{idr.d_theta_0p5m.iloc[0]:.4f}" if len(idr) else "\u2013"
    rows4.append({"Lower boundary, IC, lambda_LBC": lab4(r), "Condition met": "yes" if n == 3 else ("no" if n == 0 else f"no ({3 - n} of 3 seeds)"),
                  "Validation RMSE (x1e-3)": f"{r.val_rmse_mean*MS:.2f} \u00b1 {r.val_rmse_std*MS:.2f}", "d_theta at 0.30 m": dts, "d_theta at 0.50 m": dtb,
                  "Min FoS, test window (theta at 0.30 m / 0.50 m)": f"{r.fos_min_sensor_mean:.3f} / {r.fos_min_0p5m_mean:.3f}"})
keep(4, pd.DataFrame(rows4), "Lower-boundary and initial-condition scenarios (three seeds, one-third budget); all values on the test inputs.")

In [ ]:
# ---- Table 5: Dev107 test RMSE by strategy and number of days ----
def row5(label, vals): return {"Strategy (trained parameters)": label, **{f"{n} d": f"{v:.3f}" for n, v in zip(NDAYS, vals)}}
S5 = [("recal", "Recalibration (2)"), ("head_pinn", "Last layer, physics (65)"), ("full_pinn", "All layers, physics (76,801)"),
      ("full_data_only", "All layers, data-only (76,801)"), ("scratch_pinn", "From scratch, physics (76,801)"), ("scratch_data_only", "From scratch, data-only (76,801)")]
rows5 = [row5(lab, curve(t107, k)[0]) for k, lab in S5]
rows5.append(row5("ARX from scratch (16)", tml[tml.model == "ARX from scratch"].set_index("n_days").reindex(NDAYS).test_rmse.values * MS))
rows5.append(row5("Random forest from scratch", tml[tml.model == "RF from scratch"].groupby("n_days").test_rmse.mean().reindex(NDAYS).values * MS))
zero = lambda k: float(t107[t107.strategy == k].test_rmse.iloc[0]) * MS
print(f"Persistence {PERS107*MS:.3f}; zero-shot: {zero('zero_shot'):.3f} (physics-informed), {zero('zero_shot_data_only'):.3f} (data-only), "
      f"{float(tml[tml.model=='ARX zero-shot'].test_rmse.iloc[0])*MS:.3f} (ARX), {tml[tml.model=='RF zero-shot'].test_rmse.mean()*MS:.3f} (random forest)")
keep(5, pd.DataFrame(rows5), "Dev107 test RMSE (x1e-3 m3 m-3; 12-29 September 2026; mean of three seeds) by strategy and number of days of Dev107 data.")

In [ ]:
# ---- Table 6: external sites ----
rows6 = []
def r6(label, key, zero_key=None):
    row = {"Strategy (trained parameters)": label}
    for site, tag in (("utuado", "Utuado"), ("california", "California")):
        df = ext[site]
        row[f"{tag} 0 d"] = f"{float(df[df.strategy == zero_key].test_rmse.iloc[0]) * MS:.2f}" if zero_key else "\u2013"
        y = curve(df, key)[0] if key else [np.nan] * 4
        for n, v in zip(NDAYS, y): row[f"{tag} {n} d"] = "\u2013" if not np.isfinite(v) else f"{v:.2f}"
    return row
rows6 += [r6("Zero-shot, physics (0)", None, "zero_shot"), r6("Zero-shot, data-only (0)", None, "zero_shot_data_only")]
rows6 += [r6(lab, k) for k, lab in [("recal", "Recalibration (2)"), ("head_pinn", "Last layer, physics (65)"), ("full_pinn", "All layers, physics (76,801)"),
          ("full_data_only", "All layers, data-only (76,801)"), ("scratch_pinn", "From scratch, physics (76,801)"), ("scratch_data_only", "From scratch, data-only (76,801)")]]
keep(6, pd.DataFrame(rows6), "Test RMSE (x1e-3 m3 m-3; mean of three seeds) at the external sites, by strategy and number of days of local data.")

In [ ]:
# ---- Table 7: alert summary and RED-alert policies ----
nc = fa["TL"][fa["TL"] < TL_CAP_H]; imin = int(np.argmin(fa["TL"].values))
summary7 = [("Analysed samples / monitored days", f"{len(fa):,} / {days:.1f}"), ("FoS in analysed record (min / mean)", f"{fa['FoS_piml'].min():.3f} / {fa['FoS_piml'].mean():.3f}"),
            ("GREEN / YELLOW / RED (% of samples)", f"{(fa['TL'] > YELLOW_H).mean()*100:.2f} / {((fa['TL'] > RED_H) & (fa['TL'] <= YELLOW_H)).mean()*100:.2f} / {(fa['TL'] <= RED_H).mean()*100:.2f}"),
            ("Median lead time when FoS is falling / minimum lead time (h)", f"{nc.median():.1f} / {fa['TL'].min():.2f}"),
            ("Raw RED episodes / confirmed escalations", f"{len(e_red)} / {len(e_esc)}")]
pol = pd.read_csv(os.path.join(RC, "alert_debounce_policies.csv"))
rows7 = [{"Quantity": a, "Value": b} for a, b in summary7]
rows7.append({"Quantity": "RED-alert policy", "Value": "RED alerts (reduction vs raw); storms 18-19 Nov, 24-25 Jan; max. delay"})
EVC = [c for c in pol.columns if c.startswith("Ev")]          # showcase storms: 10 Nov, 18-19 Nov, 24-25 Jan
ev = lambda x: "kept" if x == "preserved" else x
for _, r in pol.iterrows():
    nm = (r["policy"].replace("RED persistence, 2-of-3", "RED persistence, 2 of 3").replace("RED persistence, 3-of-5", "RED persistence, 3 of 5")
          .replace("Adopted: 2-of-3 escalation + single-sample RED (30-min GREEN release)", "Adopted: 2-of-3 escalation + single RED sample"))
    red_ = f"{int(r['red_episodes'])} ({r['reduction_vs_raw_pct']:.0f}%)" if r["reduction_vs_raw_pct"] > 0 else f"{int(r['red_episodes'])} (–)"
    rows7.append({"Quantity": nm, "Value": f"{red_}; {ev(r[EVC[1]])}, {ev(r[EVC[2]])}; {r['max_confirmation_delay_min']:.0f} min"})
keep(7, pd.DataFrame(rows7), "Alert summary for the final physics-informed network, Dev108, and comparison of RED-alert policies.")

In [ ]:
# ---- all tables in one Excel file ----
try:
    with pd.ExcelWriter(os.path.join(TAB_DIR, "manuscript_tables.xlsx")) as xw:
        for n, df in TABLES.items(): df.to_excel(xw, sheet_name=f"Table {n}", index=False)
    print("saved manuscript_tables/manuscript_tables.xlsx")
except ImportError:
    print("openpyxl not installed: the CSV files were written, the Excel file was skipped")
print("\nfigures:", sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")))
print("tables :", sorted(f for f in os.listdir(TAB_DIR)))